# Lab 10: MLP Model for Time-Series Forecasting

**Student Name:** Samiullah Khan  
**Registration Number:** 22JZELE0492  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Lab Overview
This notebook develops a Multi-Layer Perceptron model for time-series forecasting. It loads preprocessed time-series data, defines an MLP architecture, configures checkpoints and monitoring callbacks, trains the model, loads saved results, and evaluates prediction performance using regression metrics.

## Learning Objectives
- Set the working directory and import the required machine learning, TensorFlow, and utility modules.
- Define an MLP neural network architecture for time-series input data.
- Configure optimizer, checkpoints, and training-monitor callbacks.
- Load train, validation, and test CSV files for forecasting.
- Train and evaluate the model using MAE, MSE, RMSE, R2, and explained variance score.

## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 working path and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utilities.


In [1]:
# Set the working directory for Lab 10/11 files.
import os
os.chdir(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11')

In [2]:
# Import metrics, time-series helpers, callbacks, TensorFlow/Keras layers, and utilities.
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
# Define training state and input dimensions for the time-series windows.
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
# Build a simple MLP model for forecasting from flattened time-series windows.
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [5]:
model1 = MLP()
model1.summary()

c:\Users\Sami\anaconda3\envs\machinelearning\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”³â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”³â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”“
â”ƒ Layer (type)                    â”ƒ Output Shape           â”ƒ       Param # â”ƒ
â”¡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â•‡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â•‡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”©
â”‚ flatten (Flatten)               â”‚ (None, 504)            â”‚             0 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ dense (Dense)                   â”‚ (None, 32)             â”‚        16,160 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ dense_1 (Dense)                 â”‚ (None, 1)              â”‚            33 â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”´â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”´â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜

 Total params: 16,193 (63.25 KB)

 Trainable params: 16,193 (63.25 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and MLP Architecture
The following cells define time steps, number of features, and the MLP model structure used for forecasting.


In [6]:
tensorflow.keras.utils.plot_model(model1)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [7]:
# Define checkpoint, output, figure, and training-history paths.
checkpoints = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
os.path.exists(JSON_PATH)

True

In [9]:
# Save only the best model checkpoint based on validation loss.
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# Group callbacks for model training.
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [10]:
# Compile a new model or load an existing checkpoint for continued training.
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # Update the learning rate before resuming training.
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## Section 3: Checkpoints, Callbacks, and Training Setup
This section configures model checkpoints, training monitor paths, optimizer settings, and compile/training parameters.


In [11]:
# Load train, validation, test, and scaler files.
import os
path_dataset =r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Sami\anaconda3\envs\machinelearning\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [12]:
# Convert raw arrays into univariate multi-step forecasting windows.
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.0 sec


In [13]:
train_X.shape

(835, 24, 21)

In [14]:
# Train the MLP model on the prepared forecasting windows.
epochs = 2
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/2
16/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 4ms/step - loss: 0.2687 - mae: 0.2687 - mape: 137.6902 
Epoch 1: val_loss improved from None to 0.11752, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0001-loss0.12.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 4s 30ms/step - loss: 0.2053 - mae: 0.2053 - mape: 104.7465 - val_loss: 0.1175 - val_mae: 0.1175 - val_mape: 38.8529
Epoch 2/2
20/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 6ms/step - loss: 0.1544 - mae: 0.1544 - mape: 80.4901  
Epoch 2: val_loss did not improve from 0.11752
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 19ms/step - loss: 0.1335 - mae: 0.1335 - mape: 69.6740 - val_loss: 0.1207 - val_mae: 0.1207 - val_mape: 38.0961


In [15]:
# Load the trained model and evaluate forecasting performance on the test set.
model = load_model(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0001-loss0.12.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 95ms/step
Mean Absolute Error (MAE): 10801.44
Median Absolute Error (MedAE): 10574.29
Mean Squared Error (MSE): 117542447.48
Root Mean Squared Error (RMSE): 10841.7
Mean Absolute Percentage Error (MAPE): 69.08 %
Median Absolute Percentage Error (MDAPE): 68.33 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Section 4: Dataset Loading and Forecast Evaluation
The remaining cells load CSV datasets, prepare forecasting windows, run prediction, and evaluate model performance using regression metrics.


## Fine Tuning

In [16]:
# Configure fine-tuning checkpoint path and starting model.
checkpoints = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0001-loss0.12.h5'
start_epoch= 56

In [18]:
# Set up fine-tuning callbacks and load the previous checkpoint.
EpochCheckpoint1 = ModelCheckpoint(
    checkpoints,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

TrainingMonitor1 = TrainingMonitor(
    FIG_PATH,
    jsonPath=JSON_PATH,
    startAt=start_epoch
)

callbacks = [EpochCheckpoint1, TrainingMonitor1]

model_path = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0001-loss0.12.h5'
# model_path = None   # use this if you want to train from scratch

if model_path is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)

    opt = Adam(learning_rate=1e-3)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    print("[INFO] learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E1-cp-0001-loss0.12.h5...
[INFO] learning rate: 9.999999747378752e-05


In [19]:
# Continue training the loaded model with the fine-tuning setup.
epochs = 5
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/5
18/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 3ms/step - loss: 0.1193 - mae: 0.1193 - mape: 60.4233 
Epoch 1: val_loss improved from None to 0.08018, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E2-cp-0001-loss0.08.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 4s 26ms/step - loss: 0.0969 - mae: 0.0969 - mape: 52.1249 - val_loss: 0.0802 - val_mae: 0.0802 - val_mape: 26.8818
Epoch 2/5
16/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 5ms/step - loss: 0.0935 - mae: 0.0935 - mape: 58.2315 
Epoch 2: val_loss improved from 0.08018 to 0.07160, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E2-cp-0002-loss0.07.h5


27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 28ms/step - loss: 0.0831 - mae: 0.0831 - mape: 45.0574 - val_loss: 0.0716 - val_mae: 0.0716 - val_mape: 24.7713
Epoch 3/5
24/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 5ms/step - loss: 0.0754 - mae: 0.0754 - mape: 39.5546  
Epoch 3: val_loss did not improve from 0.07160
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 19ms/step - loss: 0.0746 - mae: 0.0746 - mape: 40.4330 - val_loss: 0.0966 - val_mae: 0.0966 - val_mape: 31.1004
Epoch 4/5
24/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 5ms/step - loss: 0.0746 - mae: 0.0746 - mape: 39.6379
Epoch 4: val_loss did not improve from 0.07160
27/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 1s 23ms/step - loss: 0.0725 - mae: 0.0725 - mape: 38.1652 - val_loss: 0.0905 - val_mae: 0.0905 - val_mape: 27.7520
Epoch 5/5
24/27 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 5ms/step -

In [20]:
# Load the fine-tuned model and evaluate forecasting performance on the test set.
model = load_model(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 10,11\E2-cp-0002-loss0.07.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 159ms/step
Mean Absolute Error (MAE): 11067.35
Median Absolute Error (MedAE): 10785.91
Mean Squared Error (MSE): 123360235.49
Root Mean Squared Error (RMSE): 11106.77
Mean Absolute Percentage Error (MAPE): 70.77 %
Median Absolute Percentage Error (MDAPE): 69.69 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Final Conclusion
In this lab, an MLP neural network was implemented for time-series forecasting. The notebook demonstrates model creation, callback configuration, the training workflow, fine tuning, and regression-based evaluation.

## Submitted By
**Student Name:** Samiullah Khan  
**Registration Number:** 22JZELE0492  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus